In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import mean_squared_error

import lightgbm as lgb
import xgboost as xgb
import catboost as cb

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

SEEDS    = [42, 123, 456]       # seed averaging
N_FOLDS  = 10                   # 5 → 10 fold
TARGET   = 'bilissel_performans_skoru'
BASE_PATH = '/kaggle/input/competitions/yzta-2026-datathon/'

print('✅ Hazır.')

✅ Hazır.


In [2]:
# ── 1. VERİ YÜKLEME ─────────────────────────────────────────────────────────
train = pd.read_csv(BASE_PATH + 'train.csv')
test  = pd.read_csv(BASE_PATH + 'test_x.csv')
sample_sub = pd.read_csv(BASE_PATH + 'sample_submission.csv')

test_ids = test['id'].copy()
print(f'Train: {train.shape}, Test: {test.shape}')


Train: (56000, 24), Test: (24000, 23)


In [3]:
# ── 2. ÜLKE NORMALİZASYONU (eksikler tamamlandı) ────────────────────────────
ulke_map = {
    'Spain'       : 'Ispanya',
    'Germany'     : 'Almanya',
    'France'      : 'Fransa',
    'Japan'       : 'Japonya',
    'China'       : 'Cin',
    'UK'          : 'Ingiltere',
    'United Kingdom': 'Ingiltere',
    'USA'         : 'Amerika',
    'Australia'   : 'Avustralya',
    'Italy'       : 'Italya',
    'Brazil'      : 'Brezilya',
    'Canada'      : 'Kanada',
    'India'       : 'Hindistan',
    'Russia'      : 'Rusya',
    'Mexico'      : 'Meksika',
    'Portugal'    : 'Portekiz',
    'Netherlands' : 'Hollanda',
    'South Korea' : 'Guney Kore',
    'Sweden'      : 'Isvec',
    'New Zealand' : 'Yeni Zelanda',
    'Argentina'   : 'Arjantin',
}
train['ulke'] = train['ulke'].replace(ulke_map)
test['ulke']  = test['ulke'].replace(ulke_map)
print('Ülkeler:', sorted(train['ulke'].dropna().unique()))


Ülkeler: ['Amerika', 'Arjantin', 'Cin', 'Fransa', 'Guney Kore', 'Hollanda', 'Ingiltere', 'Ispanya', 'Isvec', 'Meksika', 'Portekiz', 'Yeni Zelanda']


In [4]:
# ── 3. FEATURE ENGINEERING ──────────────────────────────────────────────────
def feature_engineering(df):
    df = df.copy()

    # --- Uyku kalitesi ---
    df['kaliteli_uyku']      = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    df['hafif_uyku']         = 100 - df['kaliteli_uyku']
    df['rem_derin_oran']     = df['rem_yuzdesi'] / (df['derin_uyku_yuzdesi'] + 1e-5)
    df['log_uyku_dalma']     = np.log1p(df['uykuya_dalma_suresi_dk'])
    df['zor_uyuyan']         = (df['uykuya_dalma_suresi_dk'] > 30).astype(int)
    df['hic_uyanmayan']      = (df['gecelik_uyanma_sayisi'] == 0).astype(int)
    df['cok_uyanan']         = (df['gecelik_uyanma_sayisi'] >= 3).astype(int)
    df['uyku_bozuklugu']     = df['uykuya_dalma_suresi_dk'] * (1 + df['gecelik_uyanma_sayisi'])
    df['log_uyku_bozuklugu'] = np.log1p(df['uyku_bozuklugu'])

    # ✅ YENİ: uyku verimliliği
    df['uyku_verimlilik']    = df['rem_yuzdesi'] * df['derin_uyku_yuzdesi'] / 100.0
    df['kaliteli_saat']      = df['kaliteli_uyku'] * df.get('toplam_uyku_suresi_saat', 7) * 0.6
    df['uyku_skoru']         = df['kaliteli_uyku'] / (df['uyku_bozuklugu'] + 1)

    # --- Stres ---
    df['stres_kare']         = df['stres_skoru'] ** 2
    df['dusuk_stres']        = (df['stres_skoru'] < 4).astype(int)
    df['yuksek_stres']       = (df['stres_skoru'] > 7).astype(int)
    df['stres_log']          = np.log1p(df['stres_skoru'])

    # --- Fiziksel aktivite ---
    df['log_adim']           = np.log1p(df['gunluk_adim_sayisi'])
    df['aktif']              = (df['gunluk_adim_sayisi'] > 8000).astype(int)
    df['hareketsiz']         = (df['gunluk_adim_sayisi'] < 3000).astype(int)

    # --- Kafein & ekran ---
    df['log_kafein']         = np.log1p(df['uyku_oncesi_kafein_mg'])
    df['kafein_var']         = (df['uyku_oncesi_kafein_mg'] > 0).astype(int)
    df['log_ekran']          = np.log1p(df['uyku_oncesi_ekran_suresi_dk'])
    df['kafein_ekran']       = df['uyku_oncesi_kafein_mg'] + df['uyku_oncesi_ekran_suresi_dk'] * 0.5

    # --- BMI ---
    df['bmi_kat']            = pd.cut(df['vucut_kitle_indeksi'],
                                      bins=[0,18.5,25,30,100],
                                      labels=[0,1,2,3]).astype(float)
    df['normal_kilo']        = ((df['vucut_kitle_indeksi'] >= 18.5) &
                                (df['vucut_kitle_indeksi'] < 25)).astype(int)

    # --- Yaş ---
    df['yas_kare']           = df['yas'] ** 2
    df['yas_log']            = np.log1p(df['yas'])
    df['yas_grubu']          = pd.cut(df['yas'], bins=[0,25,35,45,55,120],
                                      labels=[0,1,2,3,4]).astype(float)

    # --- Çalışma ---
    df['fazla_mesai']        = (df['gunluk_calisma_saati'] > 9).astype(int)
    df['az_calisma']         = (df['gunluk_calisma_saati'] < 4).astype(int)

    # --- Nabız ---
    df['dusuk_nabiz']        = (df['dinlenik_nabiz_bpm'] < 60).astype(int)
    df['yuksek_nabiz']       = (df['dinlenik_nabiz_bpm'] > 80).astype(int)

    # --- Oda sıcaklığı ---
    df['optimal_sicaklik']   = ((df['oda_sicakligi_celsius'] >= 18) &
                                (df['oda_sicakligi_celsius'] <= 20)).astype(int)
    df['sicaklik_sapma']     = (df['oda_sicakligi_celsius'] - 19).abs()

    # --- Hafta sonu ---
    df['uyku_farki_abs']     = df['hafta_sonu_uyku_farki_saat'].abs()
    df['sosyal_jet_lag']     = (df['hafta_sonu_uyku_farki_saat'].abs() > 1.5).astype(int)

    # --- Şekerleme ---
    df['sekerleme_var']      = (df['sekerleme_suresi_dk'] > 0).astype(int)
    df['log_sekerleme']      = np.log1p(df['sekerleme_suresi_dk'])

    # ── ÇAPRAZ ÖZELLİKLER ──────────────────────────────────────────────────
    df['stres_x_uyku_boz']  = df['stres_skoru'] * df['uyku_bozuklugu']
    df['stres_x_kaliteli']  = df['stres_skoru'] * (100 - df['kaliteli_uyku'])
    df['adim_stres_oran']   = df['gunluk_adim_sayisi'] / (df['stres_skoru'] + 1)
    df['yas_stres']         = df['yas'] * df['stres_skoru']
    df['kafein_x_dalma']    = df['uyku_oncesi_kafein_mg'] * df['uykuya_dalma_suresi_dk']

    # ✅ YENİ çapraz özellikler
    df['stres_yas_uyku']    = df['stres_skoru'] * df['yas'] / (df['kaliteli_uyku'] + 1)
    df['kafein_yas_etki']   = df['uyku_oncesi_kafein_mg'] * np.log1p(df['yas'])
    df['toparlanma']        = df['gunluk_adim_sayisi'] / (df['stres_skoru'] + 1) / (df['yas'] + 1)
    df['uyku_stres_inv']    = df['kaliteli_uyku'] / (df['stres_skoru'] + 1)
    df['adim_x_uyku']       = df['gunluk_adim_sayisi'] * df['kaliteli_uyku'] / 100.0
    df['nabiz_stres']       = df['dinlenik_nabiz_bpm'] * df['stres_skoru']
    df['mesai_stres']       = df['gunluk_calisma_saati'] * df['stres_skoru']

    return df

train = feature_engineering(train)
test  = feature_engineering(test)
print(f'FE sonrası: Train {train.shape}, Test {test.shape}')



FE sonrası: Train (56000, 74), Test (24000, 73)


In [5]:
# ── 4. KFOLD TARGET ENCODING ─────────────────────────────────────────────────
def kfold_target_encode(train_df, test_df, cat_cols, target, n_splits=10, smooth=10):
    train_df    = train_df.copy()
    test_df     = test_df.copy()
    global_mean = train_df[target].mean()
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for col in cat_cols:
        new_col = f'{col}_te'
        train_df[new_col] = global_mean

        for tr_idx, val_idx in kf.split(train_df):
            tr    = train_df.iloc[tr_idx]
            stats = tr.groupby(col)[target].agg(['mean', 'count'])
            stats['smoothed'] = (
                (stats['mean'] * stats['count'] + global_mean * smooth) /
                (stats['count'] + smooth)
            )
            train_df.loc[train_df.index[val_idx], new_col] = (
                train_df.iloc[val_idx][col].map(stats['smoothed']).fillna(global_mean)
            )

        stats_all = train_df.groupby(col)[target].agg(['mean', 'count'])
        stats_all['smoothed'] = (
            (stats_all['mean'] * stats_all['count'] + global_mean * smooth) /
            (stats_all['count'] + smooth)
        )
        test_df[new_col] = test_df[col].map(stats_all['smoothed']).fillna(global_mean)
        print(f'  ✅ {col} → {new_col}')

    return train_df, test_df

TE_COLS = ['meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'cinsiyet', 'mevsim', 'gun_tipi']
print('Target Encoding...')
train, test = kfold_target_encode(train, test, TE_COLS, TARGET, n_splits=10, smooth=10)



Target Encoding...
  ✅ meslek → meslek_te
  ✅ ulke → ulke_te
  ✅ kronotip → kronotip_te
  ✅ ruh_sagligi_durumu → ruh_sagligi_durumu_te
  ✅ cinsiyet → cinsiyet_te
  ✅ mevsim → mevsim_te
  ✅ gun_tipi → gun_tipi_te


In [6]:
# ── 5. PREPROCESSING ─────────────────────────────────────────────────────────
train['_is_train'] = 1
test['_is_train']  = 0
test[TARGET]       = np.nan

df = pd.concat([train, test], axis=0).reset_index(drop=True)

for col in df.columns:
    if col in ['id', '_is_train', TARGET]:
        continue
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

cat_cols = ['cinsiyet', 'meslek', 'ulke', 'kronotip', 'ruh_sagligi_durumu', 'mevsim', 'gun_tipi']
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col].astype(str))

EXCLUDE      = ['id', '_is_train', TARGET]
feature_cols = [c for c in df.columns if c not in EXCLUDE]

df_train = df[df['_is_train'] == 1].reset_index(drop=True)
df_test  = df[df['_is_train'] == 0].reset_index(drop=True)

X      = df_train[feature_cols].values
y      = df_train[TARGET].values
X_test = df_test[feature_cols].values

print(f'Feature sayısı: {len(feature_cols)}')
print(f'X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}')


Feature sayısı: 79
X: (56000, 79), y: (56000,), X_test: (24000, 79)


In [7]:

# ── 6. MODEL PARAMETRELERİ ────────────────────────────────────────────────────
lgb_params = {
    'objective'        : 'regression',
    'metric'           : 'rmse',
    'n_estimators'     : 5000,
    'learning_rate'    : 0.005,       # ↓ daha iyi genelleme
    'num_leaves'       : 127,
    'subsample'        : 0.75,
    'colsample_bytree' : 0.75,
    'min_child_samples': 15,
    'reg_alpha'        : 0.05,
    'reg_lambda'        : 0.1,
    'n_jobs'           : -1,
    'verbose'          : -1,
}

xgb_params = {
    'objective'            : 'reg:squarederror',
    'eval_metric'          : 'rmse',
    'n_estimators'         : 5000,
    'learning_rate'        : 0.005,
    'max_depth'            : 6,
    'subsample'            : 0.75,
    'colsample_bytree'     : 0.75,
    'min_child_weight'     : 5,
    'early_stopping_rounds': 100,
    'tree_method'          : 'hist',
    'n_jobs'               : -1,
    'verbosity'            : 0,
}

cat_params = {
    'loss_function'        : 'RMSE',
    'iterations'           : 5000,
    'learning_rate'        : 0.005,
    'depth'                : 7,
    'early_stopping_rounds': 100,
    'random_seed'          : 42,
    'verbose'              : 0,
}

print('✅ Parametreler hazır.')



✅ Parametreler hazır.


In [8]:
# ── 7. SEED AVERAGING + 10-FOLD OOF ──────────────────────────────────────────
all_oof_lgb   = []
all_oof_xgb   = []
all_oof_cat   = []
all_test_lgb  = []
all_test_xgb  = []
all_test_cat  = []

for SEED in SEEDS:
    print(f'\n{"="*50}')
    print(f'  SEED = {SEED}')
    print(f'{"="*50}')

    lgb_params['random_state'] = SEED
    xgb_params['random_state'] = SEED
    cat_params['random_seed']  = SEED

    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

    oof_lgb   = np.zeros(len(X))
    oof_xgb   = np.zeros(len(X))
    oof_cat   = np.zeros(len(X))
    preds_lgb = np.zeros(len(X_test))
    preds_xgb = np.zeros(len(X_test))
    preds_cat = np.zeros(len(X_test))

    for fold, (trn_idx, val_idx) in enumerate(kf.split(X, y)):
        print(f'\n  Fold {fold+1}/{N_FOLDS}')

        X_tr, X_val = X[trn_idx], X[val_idx]
        y_tr, y_val = y[trn_idx], y[val_idx]

        # LightGBM
        model_lgb = lgb.LGBMRegressor(**lgb_params)
        model_lgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
                      callbacks=[lgb.early_stopping(100, verbose=False)])
        oof_lgb[val_idx]  = model_lgb.predict(X_val)
        preds_lgb        += model_lgb.predict(X_test) / N_FOLDS
        print(f'    LGB: {rmse(y_val, oof_lgb[val_idx]):.4f}')

        # XGBoost
        model_xgb = xgb.XGBRegressor(**xgb_params)
        model_xgb.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof_xgb[val_idx]  = model_xgb.predict(X_val)
        preds_xgb        += model_xgb.predict(X_test) / N_FOLDS
        print(f'    XGB: {rmse(y_val, oof_xgb[val_idx]):.4f}')

        # CatBoost
        model_cat = cb.CatBoostRegressor(**cat_params)
        model_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
        oof_cat[val_idx]  = model_cat.predict(X_val)
        preds_cat        += model_cat.predict(X_test) / N_FOLDS
        print(f'    CAT: {rmse(y_val, oof_cat[val_idx]):.4f}')

    print(f'\n  ✅ SEED={SEED} | LGB: {rmse(y, oof_lgb):.5f} | XGB: {rmse(y, oof_xgb):.5f} | CAT: {rmse(y, oof_cat):.5f}')

    all_oof_lgb.append(oof_lgb)
    all_oof_xgb.append(oof_xgb)
    all_oof_cat.append(oof_cat)
    all_test_lgb.append(preds_lgb)
    all_test_xgb.append(preds_xgb)
    all_test_cat.append(preds_cat)

# Seed ortalaması
oof_lgb_avg   = np.mean(all_oof_lgb,  axis=0)
oof_xgb_avg   = np.mean(all_oof_xgb,  axis=0)
oof_cat_avg   = np.mean(all_oof_cat,  axis=0)
preds_lgb_avg = np.mean(all_test_lgb, axis=0)
preds_xgb_avg = np.mean(all_test_xgb, axis=0)
preds_cat_avg = np.mean(all_test_cat, axis=0)

print(f'\n🌱 Seed avg | LGB: {rmse(y, oof_lgb_avg):.5f} | XGB: {rmse(y, oof_xgb_avg):.5f} | CAT: {rmse(y, oof_cat_avg):.5f}')




  SEED = 42

  Fold 1/10
    LGB: 1.2271
    XGB: 1.2200
    CAT: 1.2197

  Fold 2/10
    LGB: 1.2326
    XGB: 1.2545
    CAT: 1.2256

  Fold 3/10
    LGB: 1.2268
    XGB: 1.2249
    CAT: 1.2190

  Fold 4/10
    LGB: 1.2273
    XGB: 1.2210
    CAT: 1.2170

  Fold 5/10
    LGB: 1.2110
    XGB: 1.2076
    CAT: 1.2064

  Fold 6/10
    LGB: 1.2136
    XGB: 1.2184
    CAT: 1.2031

  Fold 7/10
    LGB: 1.2134
    XGB: 1.2326
    CAT: 1.2038

  Fold 8/10
    LGB: 1.2260
    XGB: 1.2333
    CAT: 1.2204

  Fold 9/10
    LGB: 1.2339
    XGB: 1.2303
    CAT: 1.2285

  Fold 10/10
    LGB: 1.2500
    XGB: 1.2481
    CAT: 1.2404

  ✅ SEED=42 | LGB: 1.22622 | XGB: 1.22913 | CAT: 1.21844

  SEED = 123

  Fold 1/10
    LGB: 1.1942
    XGB: 1.1887
    CAT: 1.1803

  Fold 2/10
    LGB: 1.2360
    XGB: 1.2310
    CAT: 1.2282

  Fold 3/10
    LGB: 1.2246
    XGB: 1.2203
    CAT: 1.2172

  Fold 4/10
    LGB: 1.2152
    XGB: 1.2141
    CAT: 1.2088

  Fold 5/10
    LGB: 1.2324
    XGB: 1.2278
    CAT: 1.2221

In [9]:
# ── 8. STACKING ───────────────────────────────────────────────────────────────
oof_stack  = np.column_stack([oof_lgb_avg, oof_xgb_avg, oof_cat_avg])
test_stack = np.column_stack([preds_lgb_avg, preds_xgb_avg, preds_cat_avg])

# Ridge stack
meta_ridge = Ridge(alpha=1.0)
stack_oof_ridge = cross_val_predict(
    meta_ridge, oof_stack, y,
    cv=KFold(N_FOLDS, shuffle=True, random_state=42)
)
stack_rmse_ridge = rmse(y, stack_oof_ridge)
meta_ridge.fit(oof_stack, y)
stack_preds_ridge = meta_ridge.predict(test_stack)

# ElasticNet stack
meta_enet = ElasticNet(alpha=0.01, l1_ratio=0.5)
stack_oof_enet = cross_val_predict(
    meta_enet, oof_stack, y,
    cv=KFold(N_FOLDS, shuffle=True, random_state=42)
)
stack_rmse_enet = rmse(y, stack_oof_enet)
meta_enet.fit(oof_stack, y)
stack_preds_enet = meta_enet.predict(test_stack)

# Ağırlıklı blend
rmse_arr    = np.array([rmse(y, oof_lgb_avg), rmse(y, oof_xgb_avg), rmse(y, oof_cat_avg)])
w           = (1 / rmse_arr) / (1 / rmse_arr).sum()
blend_oof   = oof_stack @ w
blend_preds = test_stack @ w
blend_rmse  = rmse(y, blend_oof)

print(f'\n🔗 Ridge  Stack OOF RMSE : {stack_rmse_ridge:.5f}')
print(f'🔗 Enet   Stack OOF RMSE : {stack_rmse_enet:.5f}')
print(f'⚖️  Blend  OOF RMSE      : {blend_rmse:.5f} | w={w.round(3)}')

# En iyi seçim
scores = {
    'ridge' : (stack_rmse_ridge, np.clip(stack_preds_ridge, 0, 10)),
    'enet'  : (stack_rmse_enet,  np.clip(stack_preds_enet,  0, 10)),
    'blend' : (blend_rmse,       np.clip(blend_preds,       0, 10)),
}
best_name = min(scores, key=lambda k: scores[k][0])
best_score, final_preds = scores[best_name]
print(f'\n🏆 Seçilen: {best_name.upper()} ({best_score:.5f})')



🔗 Ridge  Stack OOF RMSE : 1.21715
🔗 Enet   Stack OOF RMSE : 1.21738
⚖️  Blend  OOF RMSE      : 1.21902 | w=[0.332 0.333 0.334]

🏆 Seçilen: RIDGE (1.21715)


In [10]:
# ── 9. SUBMISSION ─────────────────────────────────────────────────────────────
submission = pd.DataFrame({
    'id'   : test_ids,
    TARGET : final_preds,
})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print('✅ Submission shape:', submission.shape)
print(submission.head())


✅ Submission shape: (24000, 2)
   id  bilissel_performans_skoru
0   1                   5.986635
1   2                   6.374668
2   3                   2.941825
3   4                   7.191559
4   5                   3.672343
